# 🧠 AITechFarm Knowledge Assistant — RAG-Powered Q&A System

**An intelligent, low-cost knowledge worker built with Retrieval Augmented Generation (RAG)**

---

## 📌 Overview

This project implements a **Retrieval Augmented Generation (RAG)** pipeline to power an accurate,
cost-efficient Q&A assistant for **AITechFarm**, a fictional applied-AI technology company.
The assistant answers employee questions about company products, contracts, and personnel by
grounding every response in a curated internal knowledge base — reducing hallucination and
keeping inference costs low.

## 🏗️ Pipeline Architecture

| Stage | What Happens |
|-------|-------------|
| **Part A — Chunking** | Markdown documents from the knowledge base are split into overlapping text chunks using `RecursiveCharacterTextSplitter` |
| **Part B — Embedding & Indexing** | Chunks are encoded into dense vectors with `all-MiniLM-L6-v2` (HuggingFace) and stored in a **Chroma** vector database |
| **Part C — Visualization** | Vector embeddings are projected to 2D/3D using **t-SNE** and rendered interactively with **Plotly**, color-coded by document type |
| **Part D — Retrieval & Generation** | User queries are embedded, relevant chunks are retrieved from Chroma, and `gpt-4.1-nano` generates a grounded answer via a Gradio chat UI |

## 🗂️ Knowledge Base

The synthetic knowledge base covers 4 document categories:
- 🏢 **Company** — general info about AITechFarm
- 📦 **Products** — 8 AI product lines
- 👥 **Employees** — 32 employee profiles
- 📄 **Contracts** — 32 client contracts

## 🎯 Design Goals

- **High accuracy** — answers are grounded in retrieved context, not model memory alone
- **Low cost** — uses a lightweight open-source embedding model + `gpt-4.1-nano` for generation
- **Transparency** — retrieved chunks can be inspected and visualized before being fed to the LLM

---

*Part of the `llm-engineering-journey` portfolio — documenting a hands-on transition from 16+ years of enterprise network engineering into AI/ML engineering.*

In [24]:
from dotenv import load_dotenv
import gradio as gr
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import numpy as np                     # for turning lists of embeddings into arrays
from sklearn.manifold import TSNE      # dimensionality reduction: 384D -> 2D or 3D
import plotly.graph_objects as go      # interactive plotting

In [25]:
load_dotenv(verbose=True)

True

In [26]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


## Part A — Chunking
Markdown documents from the knowledge base are split into overlapping text chunks using `RecursiveCharacterTextSplitter`.

In [27]:
splitters = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ','']
)
all_chunks = []
all_metadata = []

folder = glob.glob("knowledge-base/*")   # list of subfolder paths
print(folder)

for folder_path in folder:
    doc_type = os.path.basename(folder_path)          # e.g. "company", "products"
    md_files = sorted(glob.glob(os.path.join(folder_path, "*.md")))     # e.g knowledge-base/company/about.md

    for filename in md_files:
        with open(filename, "r", encoding="utf-8") as file:
            text = file.read()

        chunks = splitters.split_text(text)

        for chunk in chunks:
            all_chunks.append(chunk)
            all_metadata.append({"doc_type": doc_type, "source": filename})

print(f"Total chunks: {len(all_chunks)}")


['knowledge-base\\company', 'knowledge-base\\contracts', 'knowledge-base\\employees', 'knowledge-base\\products']
Total chunks: 236


## Part B — Embedding & Indexing
Chunks are encoded into dense vectors with `all-MiniLM-L6-v2` (HuggingFace) and stored in a **Chroma** vector database.

In [28]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
import os
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_texts(
    texts=all_chunks,
    embedding=embeddings,
    metadatas=all_metadata,
    persist_directory=db_name,
)

print(f"vectorstore created with: {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vectorstore created with: 236 documents


In [29]:
collections = vectorstore._collection
count = collections.count()

sample_embeddding =collections.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embeddding)
print(f"There are {count} vector with {dimensions} dimensions")

There are 236 vector with 384 dimensions


## Part C — Visualization
Vector embeddings are projected to 2D/3D using **t-SNE** and rendered interactively with **Plotly**, color-coded by document type.

In [30]:


# ---- Step 1: Pull everything back out of Chroma ----

collections = vectorstore._collection

# Ask Chroma for everything: the embeddings themselves, the original text,
# and the metadata dict (which contains doc_type) for every chunk stored
result = collections.get(include=['embeddings', 'documents', 'metadatas'])

vectors = np.array(result['embeddings'])   # shape becomes (num_chunks, 384)
texts = result['documents']                 # list[str] - the raw chunk text
metadatas = result['metadatas']             # list[dict] - one dict per chunk

print(f"Loaded {len(vectors)} vectors, each with {vectors.shape[1]} dimensions")

# ---- Step 2: Extract doc_type labels ----

doc_types = []
for metadata in metadatas:
    doc_types.append(metadata['doc_type'])

print(f"Found doc_types: {set(doc_types)}")   # sanity check - should show your 4 categories

# ---- Step 3: Assign a fixed color to each category ----

category_names = sorted(set(doc_types))   # builds the list from what's actually in your data
category_colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']  # keep a few spares

colors = []
for t in doc_types:
    position = category_names.index(t)         # find where this doc_type sits in the list
    colors.append(category_colors[position])    # use that position to pick the matching color

print(f"Assigned {len(colors)} colors")

# ---- Step 4: Build hover text for each point ----

hover_texts = []
for i in range(len(texts)):
    doc_type = doc_types[i]
    preview = texts[i][:100]                  # first 100 characters only
    hover_texts.append(f"Type: {doc_type}<br>Text: {preview}...")

print(f"Built {len(hover_texts)} hover labels")

# ---- Step 5: 2D t-SNE plot ----

tsne_2d = TSNE(n_components=2, random_state=42)
reduced_vectors_2d = tsne_2d.fit_transform(vectors)   # shape: (num_chunks, 2)

fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_vectors_2d[:, 0],     # all rows, column 0 -> x coordinates
    y=reduced_vectors_2d[:, 1],     # all rows, column 1 -> y coordinates
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=hover_texts,
    hoverinfo='text'
)])

fig_2d.update_layout(
    title='2D Chroma Vector Store Visualization',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig_2d.show()

# ---- Step 6: 3D t-SNE plot ----

tsne_3d = TSNE(n_components=3, random_state=42)
reduced_vectors_3d = tsne_3d.fit_transform(vectors)   # shape: (num_chunks, 3)

fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors_3d[:, 0],
    y=reduced_vectors_3d[:, 1],
    z=reduced_vectors_3d[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=hover_texts,
    hoverinfo='text'
)])

fig_3d.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig_3d.show()

Loaded 236 vectors, each with 384 dimensions
Found doc_types: {'contracts', 'company', 'products', 'employees'}
Assigned 236 colors
Built 236 hover labels


## Part D — Retrieval & Generation
User queries are embedded, relevant chunks are retrieved from Chroma, and the LLM generates a grounded answer via a Gradio chat UI.

In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
retriever = vectorstore.as_retriever()
llm =ChatOpenAI(temperature=0, model=MODEL, streaming=True)

In [32]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company AITechFarm.
You are chatting with a user about AITechFarm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [33]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    # Build context with source labels so the answer can reference where info came from
    context_parts = []
    for doc in docs:
        source_label = f"[Source: {doc.metadata.get('source', 'unknown')}]"
        context_parts.append(f"{source_label}\n{doc.page_content}")
    context = "\n\n".join(context_parts)

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    partial_response = ""
    response = llm.stream([SystemMessage(content=system_prompt), HumanMessage(content=question)])

    for chuck in response:
        partial_response += chuck.content
        yield partial_response

In [34]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
